# Análise dos resultados de segmentação e detecção dos óstios

## Objetivo

Este notebook realiza uma análise exploratória dos resultados da segmentação dos óstios a partir do pipeline. São apresentados gráficos e estatísticas para avaliar a qualidade das segmentações.

Resume status dos óstios, distâncias e Dice dos runs completos do pipeline refinado em treino, validação e teste.

In [ ]:
# ruff: noqa: E402
import sys
from pathlib import Path

NOTEBOOK_CWD = Path.cwd().resolve()
for candidate in (NOTEBOOK_CWD, *NOTEBOOK_CWD.parents):
    src_dir = candidate / "src"
    if src_dir.exists():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break

from utils.project.runtime.notebook_env import (
    configure_notebook_environment,  # noqa: E402
)

REPO_ROOT = configure_notebook_environment()

### Funções de apresentação
Os helpers abaixo organizam os gráficos específicos desta análise.

In [ ]:
from eda.results.view.segmentation_results_views import (
    plot_dice_distribution_by_subset,
    plot_distance_distribution_by_subset,
    plot_paired_artery_surface_distances,
    plot_status_distribution_by_subset,
    plot_success_error_by_subset,
)

In [ ]:
import pandas as pd
from IPython.display import display

from utils.comparison_utils import filter_correct_ostia_cases, load_split_results
from utils.comparison_utils.segmentation_eda import (
    additional_distance_summary_rows,
    artery_surface_distance_cases,
    build_success_status_summary_by_subset,
    pair_artery_surface_distance_cases,
    summarize_artery_surface_distance_cases,
    summarize_surface_distances_with_successful_ostia,
)
from utils.project.runtime.notebook_env import get_default_split_paths

SUCCESS_STATUS = ["both_tolerable", "both_correct"]

## Configuração

Ajuste nesta seção apenas os parâmetros da análise; o pipeline base não é alterado.

## Carregamento dos Dados
Os dados de saída do pipeline são carregados a partir do arquivo CSV gerado, contendo as informações de status e coordenadas dos óstios para cada imagem.

In [ ]:
SPLIT_PATHS_BY_RESOLUTION = get_default_split_paths(REPO_ROOT)
VALID_SPLITS = ("train", "val", "test")
RESOLUTION_KEYS = {"high": "high_res", "mid": "mid_res"}

# Carregar todos os dados diretamente pelo helper compartilhado.
data = {
    resolution: {
        split: load_split_results(
            SPLIT_PATHS_BY_RESOLUTION,
            resolution_key,
            split,
        )
        for split in VALID_SPLITS
    }
    for resolution, resolution_key in RESOLUTION_KEYS.items()
}

# Dataset atualmente ativo (padrão: mid/train)
current_resolution = "mid"
current_split = "train"
df = data[current_resolution][current_split]

available_pairs = [
    f"{resolution}/{split}"
    for resolution, split_data in data.items()
    for split, split_df in split_data.items()
    if split_df is not None
]
print("Dados carregados com sucesso!")
print("\nResolução e split disponíveis:", ", ".join(available_pairs))
print(f"Dados ativos: {current_resolution}/{current_split}")
if df is not None:
    display(df.head())
else:
    print("Referência canônica ativa ausente; nenhum outro run foi usado.")
# Torna explícita a referência canônica usada em cada grupo.
source_rows = []
for resolution_key, split_paths in SPLIT_PATHS_BY_RESOLUTION.items():
    for split in VALID_SPLITS:
        numeric = split_paths.get(split)
        source_rows.append(
            {
                "resolução": resolution_key,
                "conjunto": split,
                "referência": numeric.relative_to(REPO_ROOT).as_posix()
                if numeric
                else "ausente",
                "exames": len(
                    data["mid" if resolution_key == "mid_res" else "high"][split]
                )
                if data["mid" if resolution_key == "mid_res" else "high"][split]
                is not None
                else 0,
            }
        )
display(pd.DataFrame(source_rows))

## Análise

As subseções abaixo apresentam as métricas, tabelas ou visualizações do objetivo definido.

## Distribuição dos Status de Segmentação

### Treino (Train)

In [ ]:
plot_status_distribution_by_subset(data, "train")

### Validação (Val)

In [ ]:
plot_status_distribution_by_subset(data, "val")

### Teste (Test)

In [ ]:
plot_status_distribution_by_subset(data, "test")

## Acertos e Erros na Segmentação

### Treino (Train)

In [ ]:
plot_success_error_by_subset(data, "train", SUCCESS_STATUS)
build_success_status_summary_by_subset(data, "train", SUCCESS_STATUS)

### Validação (Val)

In [ ]:
plot_success_error_by_subset(data, "val", SUCCESS_STATUS)
build_success_status_summary_by_subset(data, "val", SUCCESS_STATUS)

### Teste (Test)

In [ ]:
plot_success_error_by_subset(data, "test", SUCCESS_STATUS)
build_success_status_summary_by_subset(data, "test", SUCCESS_STATUS)

## Distribuição das Distâncias Left e Right (mm)

A linha tracejada vermelha em **7 mm** representa o limite de tolerância adotado para considerar a detecção do óstio como aceitável.

### Treino (Train)

In [ ]:
plot_distance_distribution_by_subset(data, "train", bins=25)

### Validação (Val)

In [ ]:
plot_distance_distribution_by_subset(data, "val", bins=15)

### Teste (Test)

In [ ]:
plot_distance_distribution_by_subset(data, "test", bins=25)

## Distribuição do Dice Score

### Treino (Train)

In [ ]:
plot_dice_distribution_by_subset(
    data["mid"]["train"], data["high"]["train"], "Treino - Todos os casos"
)

In [ ]:
plot_dice_distribution_by_subset(
    filter_correct_ostia_cases(data["mid"]["train"]),
    filter_correct_ostia_cases(data["high"]["train"]),
    "Treino - Sem óstios incorretos",
)

### Validação (Val)

In [ ]:
plot_dice_distribution_by_subset(
    data["mid"]["val"], data["high"]["val"], "Validação - Todos os casos"
)

In [ ]:
plot_dice_distribution_by_subset(
    filter_correct_ostia_cases(data["mid"]["val"]),
    filter_correct_ostia_cases(data["high"]["val"]),
    "Validação - Sem óstios incorretos",
)

### Teste (Test)

In [ ]:
plot_dice_distribution_by_subset(
    data["mid"]["test"], data["high"]["test"], "Teste - Todos os casos"
)

In [ ]:
plot_dice_distribution_by_subset(
    filter_correct_ostia_cases(data["mid"]["test"]),
    filter_correct_ostia_cases(data["high"]["test"]),
    "Teste - Sem óstios incorretos",
)

## Distâncias de superfície arterial — HD e AHD

As distâncias comparam, em milímetros, as bordas da máscara arterial final
após a morfologia com as da referência arterial, na grade processada de cada
resolução. Cada ponto de uma superfície é comparado ao ponto mais próximo da
outra, nos dois sentidos. **HD** é o maior valor encontrado. **AHD** é a média
das duas médias direcionais, com o mesmo peso para cada sentido. Valores
menores indicam superfícies mais próximas. Estas métricas são distintas das
distâncias de localização dos óstios.

A tabela usa exames com status `ok` e **ambas** as distâncias válidas. Exames
sem distância ficam fora das estatísticas, inclusive os de máscara vazia.
O **Dice médio** usa os valores disponíveis nesses mesmos exames, na mesma fonte.
A seção usa os CSVs canônicos de cada resolução e conjunto.


In [ ]:
surface_sources = {
    (resolution, split): (data[resolution][split], "Canonical")
    for split in VALID_SPLITS
    for resolution in ("mid", "high")
}

surface_cases = {}
surface_summary_rows = []
for (resolution, split), (frame, source) in surface_sources.items():
    cases = artery_surface_distance_cases(frame)
    surface_cases[resolution, split] = cases
    surface_summary_rows.append(
        {
            "Conjunto": split,
            "Resolução": resolution,
            "Fonte": source if frame is not None else "Ausente",
            **summarize_artery_surface_distance_cases(frame, cases),
        }
    )
surface_summary = pd.DataFrame(surface_summary_rows)
display(
    surface_summary.round(
        {
            "HD média (mm)": 2,
            "HD mediana (mm)": 2,
            "AHD média (mm)": 2,
            "AHD mediana (mm)": 2,
            "Dice médio": 4,
        }
    ).fillna("—")
)
additional_rows = [
    {"Resolução": resolution, "Conjunto": split, "Fonte": "Canonical", **row}
    for resolution in ("mid", "high")
    for split in VALID_SPLITS
    for row in additional_distance_summary_rows(data[resolution][split])
]
if additional_rows:
    display(pd.DataFrame(additional_rows).round(2))

### HD e AHD com ambos os óstios corretos ou toleráveis

Conta como sucesso os status `both_correct` e `both_tolerable`. Nos dois
casos, cada óstio intersecta a referência arterial ou está dentro da
tolerância configurada. A coluna **Exames com HD e AHD válidos** informa
quantos casos entraram nas médias e medianas. A tabela usa as fontes e o
filtro de validade das distâncias da tabela geral.

In [ ]:
surface_successful_ostia_rows = []
for (resolution, split), (frame, source) in surface_sources.items():
    surface_successful_ostia_rows.append(
        {
            "Conjunto": split,
            "Resolução": resolution,
            "Fonte": source if frame is not None else "Ausente",
            **summarize_surface_distances_with_successful_ostia(
                frame, surface_cases[resolution, split]
            ),
        }
    )
surface_successful_ostia_summary = pd.DataFrame(surface_successful_ostia_rows)
display(surface_successful_ostia_summary.round(2).fillna("—"))

### Comparação pareada no teste

Cada ponto é um ID com HD e AHD válidos nas duas resoluções. Pontos acima da
diagonal indicam distância maior em high. As médias nos títulos usam apenas
os exames pareados e podem diferir das médias por resolução na tabela.

In [ ]:
surface_test_pairs = pair_artery_surface_distance_cases(
    surface_cases["mid", "test"], surface_cases["high", "test"]
)
plot_paired_artery_surface_distances(
    surface_test_pairs,
    surface_sources["mid", "test"][1],
    surface_sources["high", "test"][1],
)

## Conclusão

Os resumos de status, distância e Dice permitem avaliar os splits disponíveis e separar o impacto da detecção dos óstios sobre a segmentação arterial.